# Part 2 — Deep learning: from NLP output to a risk score

**Run `1_NLP_text_to_features.ipynb` first.** This notebook reads only what that one wrote into
`nlp_output/`, so the two stages are completely separate.

**What this notebook does**

1. loads the NLP output and rebuilds nothing
2. fits two gradient-boosting baselines — the bar the deep model has to clear
3. builds and trains the fusion network: FinBERT embeddings → BiGRU → attention → gated fusion
4. averages seven random seeds, then blends with the baseline
5. checks the result: label-shuffle leak test, paired tests across folds, bootstrap interval
6. answers the four pre-registered hypotheses

**Final version (3 October 2026).** Three corrections since the progress report: the GRU now reads
exactly the first 96 sentences (packed sequences; before, its backward direction also ran over the
padding), the leak test uses all five folds, and Part 1 now uses the corrected CARO flags and the
frozen folds. The numbers in the final report come from `final_run/` (`run_gpu.py` runs this same
network, seven seeds, on the same folds); a run of this notebook reproduces them up to GPU
non-determinism. Outputs are cleared: re-run on Colab.

**Runtime:** about 25 minutes on a free T4 GPU.

## 1 · Load the NLP output

In [ ]:
import json, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from google.colab import drive
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata, ttest_rel, wilcoxon
import matplotlib.pyplot as plt

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/DL data"); OUT = DATA/"nlp_output"
dev  = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then re-run."

df   = pd.read_csv(OUT/"modelling_table.csv")
meta = json.loads((OUT/"feature_columns.json").read_text())
RATIOS, LANG = meta["ratios"], meta["language"]
MAX_SENTS, DIM, N_SPLITS = meta["max_sents"], meta["dim"], meta["n_splits"]
E_mdna = np.load(OUT/"embeddings_mdna.npy")
E_aud  = np.load(OUT/"embeddings_auditor.npy")

assert df.doc_id.tolist() == meta["doc_id_order"], "table and embeddings are out of step"
assert len(df) == len(E_mdna) == len(E_aud)

y      = df.label.to_numpy().astype(np.float32)
groups = df.pair_id.to_numpy()
FOLD   = df.fold.to_numpy()
STRICT = df.strict.to_numpy().astype(bool)
# a mask marking which sentence slots are real text rather than padding
M_mdna = np.arange(MAX_SENTS)[None,:] < df.sent_len_mdna.to_numpy()[:,None]
M_aud  = np.arange(MAX_SENTS)[None,:] < df.sent_len_auditor.to_numpy()[:,None]

print(f"{len(df)} rows | {int((y==1).sum())} distressed | {df.pair_id.nunique()} pairs")
print(f"{len(RATIOS)} ratios, {len(LANG)} language features, embeddings {E_mdna.shape}")

## 2 · How performance is measured

**PR-AUC is the headline.** Precision-recall area suits an imbalanced problem where the positive
class is what matters. ROC-AUC is reported alongside.

**Per-fold mean, never a single pooled number.** Each fold trains its own network with its own
stopping point, so the five models emit probabilities on slightly different scales. Pooling all
predictions and scoring once penalises the neural models for a calibration artefact that has
nothing to do with their ability to rank companies — on this data it costs them 0.053 PR-AUC
against 0.008 for the baselines. Scoring each fold separately and averaging is the fair comparison.

`rank_norm` converts each fold's scores to within-fold ranks. It changes nothing inside a fold, but
makes two models' scores comparable when we average them together in the blend.

In [ ]:
def rank_norm(p):
    """Map each fold's predictions onto (0,1) ranks. Ordering within a fold is untouched."""
    out = np.full(len(p), np.nan)
    for k in range(N_SPLITS):
        m = (FOLD == k) & ~np.isnan(p)
        if m.sum(): out[m] = rankdata(p[m]) / (m.sum() + 1)
    return out

def per_fold(p, subset=None):
    """ROC-AUC and PR-AUC in each fold, optionally restricted to a subset of rows."""
    roc, pr = [], []
    for k in range(N_SPLITS):
        m = (FOLD == k) & ~np.isnan(p)
        if subset is not None: m = m & subset
        if m.sum() < 5 or len(set(y[m])) < 2: continue
        roc.append(roc_auc_score(y[m], p[m])); pr.append(average_precision_score(y[m], p[m]))
    return np.array(roc), np.array(pr)

def report(p, name, subset=None):
    roc, pr = per_fold(p, subset)
    print(f"  {name:32s} ROC {roc.mean():.3f}+/-{roc.std(ddof=1):.3f}   "
          f"PR {pr.mean():.3f}+/-{pr.std(ddof=1):.3f}")
    return pr

## 3 · Baselines

Two gradient-boosting models. These are what the deep model has to beat, and on small tabular data
boosted trees are a genuinely hard baseline.

- **Ratios only** — the conventional approach: eleven accounting ratios, no text at all.
- **Ratios + language** — the same plus the engineered language features from Part 1. The gap
  between these two is the contribution of classic NLP.

In [ ]:
Xr = df[RATIOS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
Xb = df[RATIOS+LANG].apply(pd.to_numeric, errors="coerce").to_numpy(float)

oof_ratio, oof_both = np.zeros(len(df)), np.zeros(len(df))
for k in range(N_SPLITS):
    tr, te = FOLD != k, FOLD == k
    oof_ratio[te] = HistGradientBoostingClassifier(random_state=0).fit(Xr[tr], y[tr]).predict_proba(Xr[te])[:,1]
    oof_both[te]  = HistGradientBoostingClassifier(random_state=0).fit(Xb[tr], y[tr]).predict_proba(Xb[te])[:,1]

print("baselines (all rows):")
report(oof_ratio, "ratios only"); report(oof_both, "ratios + language")

## 4 · The network

Three inputs arrive separately and are combined by a learned gate.

| Block | Shape in → out | What it does |
| --- | --- | --- |
| BiGRU over sentences | `96 × 768 → 96 × 64` | reads the sentence vectors in order, in both directions |
| Additive attention | `96 × 64 → 64` | weights the sentences and sums them into one vector |
| Language MLP | `53 → 64` | compresses the engineered features |
| Ratios MLP (BatchNorm) | `11 → 64` | compresses the accounting ratios |
| Softmax gate | `192 → 3` | one weight per block, per report |
| Head | `192 → 64 → 1` | produces the risk score |

**The gate is the interesting part.** Rather than concatenating the three blocks and hoping, the
network learns how much to trust each one *for each report* — so a report with no MD&A can be
down-weighted instead of dragging the prediction around. The learned weights are also readable
afterwards, which Section 9 uses.

The hyperparameters below were chosen by a nested Optuna search (small recurrent width, heavy
dropout, 96 sentences rather than 200). That search is not repeated here; Section 10 explains why
it turned out not to matter.

In [ ]:
class AdditiveAttention(nn.Module):
    """Learns a weight for every sentence, then returns their weighted sum."""
    def __init__(self, d, hidden):
        super().__init__()
        self.proj, self.v = nn.Linear(d, hidden), nn.Linear(hidden, 1, bias=False)
    def forward(self, h, mask):
        score = self.v(torch.tanh(self.proj(h))).squeeze(-1)
        score = score.masked_fill(~mask, torch.finfo(score.dtype).min)   # padding gets no weight
        w = torch.softmax(score, 1).unsqueeze(-1)
        return (h * w).sum(1), w.squeeze(-1)

def mlp(i, h, o, drop, bn=False):
    layers = [nn.Linear(i, h)]
    if bn: layers.append(nn.BatchNorm1d(h))
    return nn.Sequential(*(layers + [nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o), nn.ReLU()]))

class FusionNet(nn.Module):
    def __init__(self, n_lang, n_ratio, gru=32, block=64, attn_hidden=32, drop=0.5, dim=768):
        super().__init__()
        self.gru  = nn.GRU(dim, gru, batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(2*gru, attn_hidden)
        self.text_proj = nn.Sequential(nn.Linear(2*gru, block), nn.ReLU())
        self.lang  = mlp(n_lang,  block, block, drop)
        self.ratio = mlp(n_ratio, block, block, drop, bn=True)
        self.gate  = nn.Linear(3*block, 3)
        self.head  = nn.Sequential(nn.Linear(3*block, block), nn.ReLU(),
                                   nn.Dropout(drop), nn.Linear(block, 1))

    def forward(self, emb, mask, lang, ratio):
        # read exactly the real sentences: packing skips the zero padding, which the backward
        # direction of an unpacked GRU would otherwise run through
        lengths = mask.sum(1)
        packed = pack_padded_sequence(emb, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        h, _ = pad_packed_sequence(out, batch_first=True, total_length=mask.shape[1])
        pooled, attn_w = self.attn(h, mask)
        # a report with no usable text contributes an all-zero text block
        t = self.text_proj(pooled) * mask.any(1, keepdim=True).float()
        l, r = self.lang(lang), self.ratio(ratio)
        g = torch.softmax(self.gate(torch.cat([t, l, r], -1)), -1).unsqueeze(-1)   # (B,3,1)
        fused = (torch.stack([t, l, r], 1) * g).flatten(1)                          # (B,192)
        return self.head(fused).squeeze(-1), g.squeeze(-1), attn_w

CFG = dict(gru=32, block=64, attn_hidden=32, drop=0.50, lr=1e-3, wd=0.0,
           batch=16, gamma=1.0, n_sents=96, epochs=40, patience=8)
n_par = sum(p.numel() for p in FusionNet(len(LANG), len(RATIOS)).parameters() if p.requires_grad)
print(f"trainable parameters: {n_par:,}   (FinBERT's ~110,000,000 stay frozen)")

## 5 · Loss and training

**Focal loss.** Ordinary cross-entropy spends most of its effort on examples the model already gets
right. Focal loss down-weights those and concentrates on the hard ones, which suits a problem where
the interesting companies are the ambiguous ones. A small amount of plain cross-entropy is kept in
the mix to stabilise it.

**Preprocessing happens inside the loop.** Medians for filling gaps and the mean/standard deviation
for scaling are computed from the **training fold only**, then applied to the test fold. Computing
them over the whole dataset would leak test-set information into training — the same mistake as
fitting the perplexity language model globally.

**Early stopping** uses an inner split of the training data, so the test fold never influences when
training stops.

In [ ]:
def focal_bce(logit, target, gamma=1.0, ce_weight=0.1):
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    if gamma == 0: return ce.mean()
    return ((1 - torch.exp(-ce))**gamma * ce).mean() + ce_weight * ce.mean()

def prep(train_idx, idx, X):
    """Fill gaps with the TRAINING median and scale by TRAINING mean/sd."""
    med = np.nanmedian(X[train_idx], 0); med = np.where(np.isnan(med), 0, med)
    mu  = np.nanmean(X[train_idx], 0);   sd = np.nanstd(X[train_idx], 0)
    mu  = np.where(np.isnan(mu), 0, mu)
    sd  = np.where(~np.isfinite(sd) | (sd < 1e-6), 1.0, sd)
    return ((np.where(np.isnan(X[idx]), med, X[idx]) - mu) / sd).astype(np.float32)

Xl_all = df[LANG].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
Xr_all = df[RATIOS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)

def train_fold(k, E, MASK, labels=None, cfg=CFG, seed=0, keep_model=False):
    yy = y if labels is None else labels
    te = np.flatnonzero(FOLD == k); dv = np.flatnonzero(FOLD != k)
    a, b = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed)
                .split(dv, yy[dv], groups[dv]))
    tr, va = dv[a], dv[b]                      # inner split, also grouped by pair
    torch.manual_seed(seed); np.random.seed(seed)
    ns = cfg["n_sents"]

    def dataset(idx):
        # only the first ns sentence slots go in; the mask marks the real sentences among them
        return TensorDataset(torch.from_numpy(np.ascontiguousarray(E[idx, :ns]).astype(np.float32)),
                             torch.from_numpy(np.ascontiguousarray(MASK[idx, :ns])),
                             torch.from_numpy(prep(tr, idx, Xl_all)),
                             torch.from_numpy(prep(tr, idx, Xr_all)),
                             torch.from_numpy(yy[idx]))
    Ltr = DataLoader(dataset(tr), batch_size=cfg["batch"], shuffle=True, drop_last=True)
    Lva = DataLoader(dataset(va), batch_size=128)
    Lte = DataLoader(dataset(te), batch_size=128)

    model = FusionNet(Xl_all.shape[1], Xr_all.shape[1],
                      cfg["gru"], cfg["block"], cfg["attn_hidden"], cfg["drop"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])

    def infer(loader):
        model.eval(); out = []
        with torch.no_grad():
            for e, m, l, r, _ in loader:
                lo, _, _ = model(e.to(dev), m.to(dev), l.to(dev), r.to(dev))
                out.append(torch.sigmoid(lo).cpu().numpy())
        return np.concatenate(out)

    best, bad, best_state = -1.0, 0, None
    for _ in range(cfg["epochs"]):
        model.train()
        for e, m, l, r, t in Ltr:
            opt.zero_grad()
            lo, _, _ = model(e.to(dev), m.to(dev), l.to(dev), r.to(dev))
            focal_bce(lo, t.to(dev), cfg["gamma"]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        v = average_precision_score(yy[va], infer(Lva))
        if v > best:
            best, bad = v, 0
            best_state = {kk: vv.detach().clone() for kk, vv in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    model.load_state_dict(best_state)
    return (infer(Lte), te, model, tr) if keep_model else (infer(Lte), te)

def run_cv(E, MASK, labels=None, cfg=CFG, seed=0):
    oof = np.full(len(df), np.nan)
    for k in range(N_SPLITS):
        pred, te = train_fold(k, E, MASK, labels, cfg, seed); oof[te] = pred
    return oof
print("ready")

## 6 · Train — seven seeds

A single run of a network this size on 763 rows is unstable: which epoch early stopping picks
depends heavily on a ~120-row inner split. Training the same model under seven random seeds and
averaging the ranks is the cheapest fix, and in testing it cut the fold-to-fold spread from 0.090
to 0.064.

In [ ]:
seeds, per_seed = list(range(7)), []
t0 = time.time()
for s in seeds:
    o = run_cv(E_mdna, M_mdna, seed=s); per_seed.append(o)
    print(f"  seed {s}: PR {per_fold(o)[1].mean():.3f}   ({time.time()-t0:.0f}s)")
oof_dl = np.nanmean([rank_norm(o) for o in per_seed], 0)
print()
report(oof_dl, "deep model (7-seed ensemble)")

## 7 · The blend

The neural model and the gradient booster make different mistakes, so averaging them helps. The
weight is fixed at 50/50 **in advance** — nothing is fitted, so there is no risk of choosing a
weight that flatters the test data. The sweep below is printed only to show the result is not
balanced on a knife edge.

In [ ]:
blend = 0.5*rank_norm(oof_dl) + 0.5*rank_norm(oof_both)

print("all 763 rows:")
for nm, p in [("ratios only", oof_ratio), ("ratios + language", oof_both),
              ("deep model", oof_dl), ("BLEND", blend)]: report(p, nm)
print("\nstrict matched subset (609 rows):")
for nm, p in [("ratios only", oof_ratio), ("ratios + language", oof_both),
              ("deep model", oof_dl), ("BLEND", blend)]: report(p, nm, STRICT)

print("\nsensitivity to the blend weight (not tuned - shown for transparency):")
for w in [0, 0.25, 0.5, 0.75, 1.0]:
    print(f"  weight on deep model = {w:.2f}  ->  PR "
          f"{per_fold(w*rank_norm(oof_dl)+(1-w)*rank_norm(oof_both))[1].mean():.3f}")

## 8 · Is the result real?

Three checks, in order of importance.

**The leak test.** Shuffle the labels so they carry no information, retrain, and confirm the model
collapses to chance. If it does not, something in the features encodes the answer and every other
number here is meaningless. This is the first question an examiner asks.

**Paired tests across folds.** With five paired observations the smallest p-value the Wilcoxon
signed-rank test can produce is **0.0625** — so a reported 0.062 means *every fold agreed*, which
is the strongest available outcome, not a near miss.

**Bootstrap.** Resample matched pairs with replacement to put an interval on the difference.

In [ ]:
print("LEAK TEST - shuffled labels, all 5 folds")
y_shuffled = y[np.random.default_rng(0).permutation(len(y))]
sh = np.full(len(df), np.nan)
for k in range(N_SPLITS):
    pred, te = train_fold(k, E_mdna, M_mdna, labels=y_shuffled,
                          cfg=dict(CFG, epochs=15, patience=5)); sh[te] = pred
m = ~np.isnan(sh)
roc_sh = roc_auc_score(y_shuffled[m], sh[m])
print(f"  shuffled-label ROC-AUC {roc_sh:.3f}")
print("  PASS - nothing encodes the outcome" if roc_sh < 0.60 else "  *** FAIL - investigate ***")

In [ ]:
print("\nPAIRED TESTS across the five folds (PR-AUC)")
models = {"ratios only": oof_ratio, "ratios + language": oof_both,
          "deep model": oof_dl, "BLEND": blend}
for label, subset in [("all 763", None), ("strict 609", STRICT)]:
    print(f"\n  --- {label} ---")
    pf = {k: per_fold(v, subset)[1] for k, v in models.items()}
    for b in ["ratios only", "ratios + language", "deep model"]:
        d = pf["BLEND"] - pf[b]
        print(f"    BLEND vs {b:20s} {d.mean():+.3f}   t p={ttest_rel(pf['BLEND'],pf[b]).pvalue:.3f}"
              f"   wilcoxon p={wilcoxon(pf['BLEND'],pf[b]).pvalue:.3f}   wins {int((d>0).sum())}/5")

print("\nBOOTSTRAP over matched pairs (2,000 resamples), BLEND vs ratios + language")
A, B = rank_norm(blend), rank_norm(oof_both)
pairs = np.array(sorted(set(groups))); idx_of = {p: np.flatnonzero(groups==p) for p in pairs}
rng = np.random.default_rng(0); d_pr = []
for _ in range(2000):
    take = np.concatenate([idx_of[p] for p in rng.choice(pairs, len(pairs), replace=True)])
    if len(set(y[take])) < 2: continue
    d_pr.append(average_precision_score(y[take],A[take]) - average_precision_score(y[take],B[take]))
d_pr = np.array(d_pr)
print(f"  PR-AUC difference {average_precision_score(y,A)-average_precision_score(y,B):+.3f}"
      f"   95% CI [{np.percentile(d_pr,2.5):+.3f}, {np.percentile(d_pr,97.5):+.3f}]"
      f"   P(>0)={(d_pr>0).mean():.2f}")

## 9 · What the model actually used

The gate weights say how much each of the three inputs the network relied on. The attention weights
say which sentences it read. Both are worth reporting whatever they show.

In [ ]:
pred, te, model, tr = train_fold(0, E_mdna, M_mdna, keep_model=True)
model.eval(); G, A_w = [], []
with torch.no_grad():
    for a in range(0, len(te), 64):
        i = te[a:a+64]
        ns = CFG["n_sents"]
        _, g, aw = model(torch.from_numpy(np.ascontiguousarray(E_mdna[i, :ns]).astype(np.float32)).to(dev),
                         torch.from_numpy(np.ascontiguousarray(M_mdna[i, :ns])).to(dev),
                         torch.from_numpy(prep(tr,i,Xl_all)).to(dev),
                         torch.from_numpy(prep(tr,i,Xr_all)).to(dev))
        G.append(g.cpu().numpy()); A_w.append(aw.cpu().numpy())
G, A_w = np.concatenate(G), np.concatenate(A_w)

print("average gate weight per input block:")
for nm, v in zip(["text (FinBERT)", "language features", "accounting ratios"], G.mean(0)):
    print(f"  {nm:22s} {v:.3f}")
plt.figure(figsize=(4.5,3))
plt.bar(["text","language\nfeatures","ratios"], G.mean(0))
plt.ylabel("average gate weight"); plt.title("What the network relied on"); plt.tight_layout(); plt.show()

> **Result in the final run: the text gate averages 0.23**, the language features 0.49 and the
> ratios 0.27 (one fold here gives a similar split). The progress-report network gave text 0.07; the
> corrected network uses it a little more, but removing the stream does not hurt (PR-AUC 0.789
> without it, 0.783 with it), and the attention is close to uniform (normalised entropy 0.98). The
> earlier observation that attention landed on boilerplate does not survive the correction: the
> network never learned to single out sentences.
>
> The language contribution comes from the **engineered features**, above all the auditor's CARO
> flags, not from a transformer reading sentences. With about 490 training reports per fold this is
> the expected outcome.

## 10 · The four pre-registered hypotheses

These were fixed in advance in the progress report, each with a stated outcome in either direction.

In [ ]:
print("H3 - does the warning appear early, or only just before the event?")
print(f"  {'horizon':28s} {'n':>5s} {'ratios only':>12s} {'full system':>12s}")
for h in ["t-1","t-2","t-3"]:
    sub = (df.horizon == h).to_numpy()
    if sub.sum() < 20: continue
    print(f"  {h:28s} {int(sub.sum()):5d} {per_fold(oof_ratio,sub)[1].mean():12.3f}"
          f" {per_fold(blend,sub)[1].mean():12.3f}")
print("  -> a flat profile means the signal survives three years before admission")

In [ ]:
print("\nH4 - is the signal only the auditor's flags?")
# up to 96 MD&A sentences followed DIRECTLY by up to 96 auditor sentences: the packed GRU reads
# the first `length` slots, so the real sentences must be contiguous
L_m = np.minimum(M_mdna.sum(1), 96); L_a = np.minimum(M_aud.sum(1), 96)
E_both = np.zeros_like(E_mdna)
for i in range(len(df)):
    E_both[i, :L_m[i]] = E_mdna[i, :L_m[i]]
    E_both[i, L_m[i]:L_m[i] + L_a[i]] = E_aud[i, :L_a[i]]
M_both = np.arange(MAX_SENTS)[None, :] < (L_m + L_a)[:, None]
for nm, EE, MM, ns in [("management discussion", E_mdna, M_mdna,  96),
                       ("auditor's report",      E_aud,  M_aud,   96),
                       ("both sections",         E_both, M_both, 192)]:
    o = run_cv(EE, MM, cfg=dict(CFG, n_sents=ns))
    report(o, nm)
print("  -> final run: the auditor's report is the stronger text (text-only PR-AUC 0.679 against 0.547)")

In [ ]:
print("\nWhere language matters most: companies whose accounts look healthy")
z = pd.to_numeric(df["altman_z_em"], errors="coerce")
safe = ((z > z.median()) & z.notna()).to_numpy()
print(f"  {int(safe.sum())} company-years with above-median Altman Z, "
      f"{int(y[safe].sum())} of them distressed (base rate {y[safe].mean():.2f})")
for nm, p in [("ratios only",oof_ratio),("ratios + language",oof_both),("BLEND",blend)]:
    report(p, nm, safe)
print("  -> compare these gaps with the full-sample gaps in Section 7")

## 11 · Summary (final run, corrected flags and network)

| Hypothesis | Verdict |
| --- | --- |
| **H1** report language separates insolvent from healthy firms | supported: ROC-AUC 0.737 from the engineered features, about 0.70 from the auditor's report alone; the MD&A alone only 0.55 to 0.59 |
| **H2** language adds information beyond the ratios | supported: the full system +0.059 PR-AUC over ratios (0.807 against 0.747), all five folds, pair-bootstrap interval +0.009 to +0.105 |
| **H3** the signal survives further from the event | supported: full system 0.842, 0.814 and 0.823 at t-1, t-2 and t-3; language adds most two years out |
| **H4** the signal is not only the auditor's flags | supported in direction: the MD&A features improve on the ratios in every fold; the auditor's report is the stronger text, alone and on top of the ratios |

Headline: **PR-AUC 0.838 on the strict matched sample against 0.786 for ratios alone**, in all five
folds. On companies whose Altman Z″ is above the median the gain is **+0.164** (0.399 to 0.563),
about three times the gain on the full sample.

Reported alongside: the FinBERT stream does not help (0.789 without it, 0.783 with it), perplexity
carries no signal (ROC-AUC 0.503), year-on-year drift adds nothing, and hyperparameter search is
uninformative at this sample size (r = +0.07 between inner and test scores). Every number here is
in the final report, generated from saved predictions.